# TIR Experiment Results

Plots and tables for CS224R final project.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 150

EVAL_DIR = '/Users/sbfisher/Stanford/CS224R/final_project/eval_results'

IndentationError: unexpected indent (3193640399.py, line 11)

## 1. SFT Ablation Results

In [5]:
def pass_at_k(n, c, k):
    """Unbiased pass@k estimator."""
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

def compute_pass_at_k(scores_list, k_values=[1, 4, 16]):
    """Compute unbiased pass@k from a list of per-problem score lists."""
    results = {}
    for k in k_values:
        vals = []
        for scores in scores_list:
            n = len(scores)
            c = sum(1 for s in scores if s >= 1.0)
            if k <= n:
                vals.append(pass_at_k(n, c, k))
        results[f'pass@{k}'] = np.mean(vals) if vals else 0.0
    return results

def load_eval_results(eval_dir):
    """Load all JSONL eval result files."""
    rows = []
    for f in sorted(os.listdir(eval_dir)):
        if not f.endswith('.json'):
            continue
        path = os.path.join(eval_dir, f)
        with open(path) as fh:
            lines = [json.loads(l) for l in fh if l.strip()]
        if not lines or 'scores' not in lines[0]:
            continue
        scores_list = [l['scores'] for l in lines]
        metrics = compute_pass_at_k(scores_list)
        metrics['name'] = f.replace('.json', '')
        metrics['n_problems'] = len(lines)
        metrics['n_samples'] = len(lines[0]['scores'])
        rows.append(metrics)
    return pd.DataFrame(rows)

all_results = load_eval_results(EVAL_DIR)
all_results

FileNotFoundError: [Errno 2] No such file or directory: 'eval_results'

In [ ]:
# SFT ablation subset — v2 results only
sft_names = [
    '3tool_from_sft_v2_with_tools', '3tool_from_sft_v2_no_tools',
    'calc_only_from_sft_v2_with_tools', 'calc_only_from_sft_v2_no_tools',
    '3tool_from_base_v2_with_tools', '3tool_from_base_v2_no_tools',
    'calc_only_from_base_v2_with_tools', 'calc_only_from_base_v2_no_tools',
    'sft_eval_run', 'downloaded_eval_results',
]
sft_df = all_results[all_results['name'].isin(sft_names)].copy()

# Parse model attributes
def parse_sft_name(name):
    if name in ('sft_eval_run', 'downloaded_eval_results'):
        return {'tools_trained': 'none', 'warm_start': 'N/A',
                'eval_tools': 'no', 'label': 'Vanilla SFT'}
    tools_trained = '3tool' if '3tool' in name else 'calc_only'
    warm_start = 'from_sft' if 'from_sft' in name else 'from_base'
    eval_tools = 'with' if 'with_tools' in name else 'no'
    label = f"{tools_trained} {warm_start}"
    return {'tools_trained': tools_trained, 'warm_start': warm_start,
            'eval_tools': eval_tools, 'label': label}

attrs = pd.DataFrame([parse_sft_name(n) for n in sft_df['name']])
sft_df = pd.concat([sft_df.reset_index(drop=True), attrs], axis=1)
sft_df[['label', 'eval_tools', 'pass@1', 'pass@4', 'pass@16']].sort_values('pass@1', ascending=False)

In [ ]:
# Pivot table: model vs eval mode
sft_pivot = sft_df.pivot_table(
    index='label', columns='eval_tools', values='pass@1'
).rename(columns={'with': 'With Tools', 'no': 'No Tools'})
sft_pivot = sft_pivot.sort_values('With Tools', ascending=False)
print("SFT Ablation — pass@1 (unbiased estimator, n=50, k=16)")
print("=" * 55)
display(sft_pivot.style.format('{:.1%}').set_caption('pass@1 by Model and Eval Mode'))

In [ ]:
# Overlapping bar chart: pass@1, pass@4, pass@16 for With Tools vs No Tools
# Wider bars behind narrower ones so all three k values are visible

k_metrics = ['pass@16', 'pass@4', 'pass@1']  # draw widest (highest) first
bar_widths = [0.35, 0.25, 0.15]
alphas = [0.45, 0.65, 1.0]

# Pivot for each k
pivots = {}
for k in ['pass@1', 'pass@4', 'pass@16']:
    piv = sft_df.pivot_table(index='label', columns='eval_tools', values=k)
    piv = piv.rename(columns={'with': 'With Tools', 'no': 'No Tools'})
    pivots[k] = piv

# Use pass@1 With Tools ordering
order = pivots['pass@1'].sort_values('With Tools', ascending=False).index
for k in pivots:
    pivots[k] = pivots[k].reindex(order)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(order))
half = 0.22  # offset between With/No groups

colors_with = '#2196F3'
colors_no = '#FF9800'

for metric, w, alpha in zip(k_metrics, bar_widths, alphas):
    piv = pivots[metric]
    ax.bar(x - half, piv['With Tools'] * 100, w, color=colors_with, alpha=alpha, edgecolor='white', linewidth=0.5)
    if 'No Tools' in piv.columns:
        vals = piv['No Tools'].fillna(0) * 100
        ax.bar(x + half, vals, w, color=colors_no, alpha=alpha, edgecolor='white', linewidth=0.5)

# Value labels for pass@1 only (smallest/front bars)
piv1 = pivots['pass@1']
for i, lbl in enumerate(order):
    v = piv1.loc[lbl, 'With Tools']
    if not np.isnan(v):
        ax.text(i - half, v * 100 + 1, f'{v*100:.1f}', ha='center', va='bottom', fontsize=7)
    if 'No Tools' in piv1.columns:
        v = piv1.loc[lbl, 'No Tools']
        if not np.isnan(v):
            ax.text(i + half, v * 100 + 1, f'{v*100:.1f}', ha='center', va='bottom', fontsize=7)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=colors_with, alpha=1.0, label='With Tools'),
    Patch(facecolor=colors_no, alpha=1.0, label='No Tools'),
    Patch(facecolor='gray', alpha=0.45, label='pass@16'),
    Patch(facecolor='gray', alpha=0.65, label='pass@4'),
    Patch(facecolor='gray', alpha=1.0, label='pass@1'),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=10, ncol=2)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('SFT Ablation: Tool Access at Evaluation (pass@1/4/16)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=35, ha='right', fontsize=9)
ax.set_ylim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/sft_ablation_bar.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Line plot: pass@k vs k for each SFT ablation
k_values = [1, 4, 16]

# Build lines: one per (label, eval_tools) combo
fig, ax = plt.subplots(figsize=(10, 6))

# Distinct styles for with/no tools
style_map = {'with': '-o', 'no': '--s'}
cmap = plt.cm.tab10

# Get unique labels ordered by pass@1 with tools
labels_ordered = sft_df[sft_df['eval_tools'] == 'with'].sort_values('pass@1', ascending=False)['label'].tolist()
# Add labels that only appear in no-tools (e.g. Vanilla SFT)
for lbl in sft_df['label'].unique():
    if lbl not in labels_ordered:
        labels_ordered.append(lbl)

for i, label in enumerate(labels_ordered):
    color = cmap(i)
    for eval_mode in ['with', 'no']:
        row = sft_df[(sft_df['label'] == label) & (sft_df['eval_tools'] == eval_mode)]
        if row.empty:
            continue
        row = row.iloc[0]
        vals = [row['pass@1'] * 100, row['pass@4'] * 100, row['pass@16'] * 100]
        suffix = ' (tools)' if eval_mode == 'with' else ' (no tools)'
        ax.plot(k_values, vals, style_map[eval_mode], color=color,
                label=f'{label}{suffix}', markersize=6, linewidth=1.5)

ax.set_xlabel('k', fontsize=12)
ax.set_ylabel('pass@k (%)', fontsize=12)
ax.set_title('SFT Ablation: pass@k vs k', fontsize=14)
ax.set_xticks(k_values)
ax.set_xticklabels([str(k) for k in k_values])
ax.set_ylim(0, 100)
ax.legend(fontsize=8, ncol=2, frameon=False, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/sft_passk_lines.png', bbox_inches='tight', dpi=300)
plt.show()

## 2. RLOO Evaluation Results

In [ ]:
# RLOO evaluation results
rloo_names = [n for n in all_results['name'] if n.startswith('rloo_')]
rloo_df = all_results[all_results['name'].isin(rloo_names)].copy()

def parse_rloo_name(name):
    # e.g. rloo_vanilla_step50_with_tools, rloo_nosc_latest_no_tools
    eval_tools = 'with' if 'with_tools' in name else 'no'
    base = name.replace('_with_tools', '').replace('_no_tools', '')
    # Pretty labels
    label_map = {
        'rloo_vanilla_step50': 'Vanilla (step 50)',
        'rloo_vanilla_latest': 'Vanilla (latest)',
        'rloo_nosc_step50': 'Hierarchical (step 50)',
        'rloo_nosc_latest': 'Hierarchical (latest)',
        'rloo_sc_step50': 'Self-Critic (step 50)',
    }
    label = label_map.get(base, base)
    return {'eval_tools': eval_tools, 'label': label}

attrs = pd.DataFrame([parse_rloo_name(n) for n in rloo_df['name']])
rloo_df = pd.concat([rloo_df.reset_index(drop=True), attrs], axis=1)
rloo_df[['label', 'eval_tools', 'pass@1', 'pass@4', 'pass@16']].sort_values('pass@1', ascending=False)

In [ ]:
# Line plot: pass@k vs k for each RLOO model
k_values = [1, 4, 16]

fig, ax = plt.subplots(figsize=(10, 6))

style_map = {'with': '-o', 'no': '--s'}
cmap = plt.cm.tab10

labels_ordered = rloo_df[rloo_df['eval_tools'] == 'with'].sort_values('pass@1', ascending=False)['label'].tolist()
for lbl in rloo_df['label'].unique():
    if lbl not in labels_ordered:
        labels_ordered.append(lbl)

for i, label in enumerate(labels_ordered):
    color = cmap(i)
    for eval_mode in ['with', 'no']:
        row = rloo_df[(rloo_df['label'] == label) & (rloo_df['eval_tools'] == eval_mode)]
        if row.empty:
            continue
        row = row.iloc[0]
        vals = [row['pass@1'] * 100, row['pass@4'] * 100, row['pass@16'] * 100]
        suffix = ' (tools)' if eval_mode == 'with' else ' (no tools)'
        ax.plot(k_values, vals, style_map[eval_mode], color=color,
                label=f'{label}{suffix}', markersize=6, linewidth=1.5)

ax.set_xlabel('k', fontsize=12)
ax.set_ylabel('pass@k (%)', fontsize=12)
ax.set_title('RLOO Evaluation: pass@k vs k', fontsize=14)
ax.set_xticks(k_values)
ax.set_xticklabels([str(k) for k in k_values])
ax.set_ylim(0, 100)
ax.legend(fontsize=8, ncol=2, frameon=False, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/rloo_passk_lines.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# RLOO bar chart: pass@1/4/16, With Tools vs No Tools (overlapping bars)
from matplotlib.patches import Patch

k_metrics = ['pass@16', 'pass@4', 'pass@1']
bar_widths = [0.35, 0.25, 0.15]
alphas = [0.45, 0.65, 1.0]

pivots_rloo = {}
for k in ['pass@1', 'pass@4', 'pass@16']:
    piv = rloo_df.pivot_table(index='label', columns='eval_tools', values=k)
    piv = piv.rename(columns={'with': 'With Tools', 'no': 'No Tools'})
    pivots_rloo[k] = piv

# Order by pass@1 With Tools (descending)
has_with = 'With Tools' in pivots_rloo['pass@1'].columns
sort_col = 'With Tools' if has_with else 'No Tools'
order = pivots_rloo['pass@1'].sort_values(sort_col, ascending=False, na_position='last').index
for k in pivots_rloo:
    pivots_rloo[k] = pivots_rloo[k].reindex(order)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(order))
half = 0.22

colors_with = '#2196F3'
colors_no = '#FF9800'

for metric, w, alpha in zip(k_metrics, bar_widths, alphas):
    piv = pivots_rloo[metric]
    if 'With Tools' in piv.columns:
        ax.bar(x - half, piv['With Tools'].fillna(0) * 100, w, color=colors_with, alpha=alpha, edgecolor='white', linewidth=0.5)
    if 'No Tools' in piv.columns:
        ax.bar(x + half, piv['No Tools'].fillna(0) * 100, w, color=colors_no, alpha=alpha, edgecolor='white', linewidth=0.5)

# Value labels for pass@1
piv1 = pivots_rloo['pass@1']
for i, lbl in enumerate(order):
    if 'With Tools' in piv1.columns and not pd.isna(piv1.loc[lbl, 'With Tools']):
        v = piv1.loc[lbl, 'With Tools']
        ax.text(i - half, v * 100 + 1, f'{v*100:.1f}', ha='center', va='bottom', fontsize=7)
    if 'No Tools' in piv1.columns and not pd.isna(piv1.loc[lbl, 'No Tools']):
        v = piv1.loc[lbl, 'No Tools']
        ax.text(i + half, v * 100 + 1, f'{v*100:.1f}', ha='center', va='bottom', fontsize=7)

legend_elements = [
    Patch(facecolor=colors_with, alpha=1.0, label='With Tools'),
    Patch(facecolor=colors_no, alpha=1.0, label='No Tools'),
    Patch(facecolor='gray', alpha=0.45, label='pass@16'),
    Patch(facecolor='gray', alpha=0.65, label='pass@4'),
    Patch(facecolor='gray', alpha=1.0, label='pass@1'),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=10, ncol=2)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('RLOO Evaluation: Tool Access at Evaluation (pass@1/4/16)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=35, ha='right', fontsize=9)
ax.set_ylim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/rloo_eval_bar.png', bbox_inches='tight', dpi=300)
plt.show()